# 04 GPT Adjudication

This notebook demonstrates the post-threshold GPT decision layer for the Alibaba-container anomaly pipeline. The FiLM autoencoder remains the anomaly detector. GPT is invoked only after the reconstruction score crosses threshold, where it interprets the anomaly, filters likely false positives, assigns severity, and proposes a recommended action in structured JSON.


In [1]:
from __future__ import annotations

from ast import literal_eval
from pathlib import Path
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from container_ad_pipeline.config import PipelineConfig, ensure_pipeline_directories
from container_ad_pipeline.dataset import build_dataset_from_raw_archives, load_dataset_bundle
from container_ad_pipeline.evaluate import (
    add_top_k_feature_columns,
    evaluate_predictions,
    inject_synthetic_anomalies,
    run_model_inference,
    save_evaluation_outputs,
)
from container_ad_pipeline.gpt_adjudicator import GPTAdjudicator
from container_ad_pipeline.train import train_film_autoencoder, select_split
from container_ad_pipeline.utils import save_json

pd.set_option("display.max_columns", 200)
config = PipelineConfig()
ensure_pipeline_directories(config)

config.dataset.max_container_meta_files = 10
config.dataset.max_container_usage_files = 10
config.dataset.max_machine_meta_files = None
config.dataset.max_machine_usage_files = 10
config.dataset.max_usage_rows = 10_000
config.dataset.max_machine_usage_rows = 20_000
config.dataset.max_containers = 50
config.dataset.chunksize = 5_000

if not (config.paths.processed_dir / "X_all.npy").exists():
    build_dataset_from_raw_archives(config.paths, config.dataset, config.paths.processed_dir)

bundle = load_dataset_bundle(config.paths.processed_dir)
config.train.epochs = 30
config.train.patience = 8
config.train.batch_size = 128
if not config.paths.checkpoint_path.exists():
    train_film_autoencoder(
        bundle=bundle,
        train_config=config.train,
        checkpoint_path=config.paths.checkpoint_path,
        x_scaler_path=config.paths.x_scaler_path,
        c_scaler_path=config.paths.c_scaler_path,
        detector_meta_json=config.paths.detector_meta_json,
        detector_meta_joblib=config.paths.detector_meta_joblib,
    )

predictions_path = config.paths.evaluation_dir / config.eval.save_predictions_filename
if not predictions_path.exists():
    x_scaler = joblib.load(config.paths.x_scaler_path)
    c_scaler = joblib.load(config.paths.c_scaler_path)
    threshold_bootstrap = float(joblib.load(config.paths.detector_meta_joblib)["threshold"])
    X_test, _, _ = select_split(bundle, "test")
    X_injected, y_true, events = inject_synthetic_anomalies(
        X_test,
        feature_columns=bundle.feature_meta["feature_columns"],
        anomaly_ratio=config.eval.anomaly_ratio,
        seed=config.eval.anomaly_seed,
    )
    predictions_bootstrap = run_model_inference(
        bundle=bundle,
        model_path=config.paths.checkpoint_path,
        x_scaler=x_scaler,
        c_scaler=c_scaler,
        split_name="test",
        injected_X=X_injected,
    )
    predictions_bootstrap = add_top_k_feature_columns(predictions_bootstrap, bundle.feature_meta["feature_columns"], config.eval.top_k_features)
    artifacts_bootstrap = evaluate_predictions(predictions_bootstrap, y_true, threshold_bootstrap, config.eval, events)
    save_evaluation_outputs(artifacts_bootstrap, config.paths.evaluation_dir, config.eval)

detector_meta = joblib.load(config.paths.detector_meta_joblib)
threshold = float(detector_meta["threshold"])
feature_columns = bundle.feature_meta["feature_columns"]
adjudicator = GPTAdjudicator(config.gpt)

print("Threshold:", threshold)
print("Predictions file:", predictions_path)


Threshold: 5.500068187713623
Predictions file: C:\Users\kaspe\Desktop\Project\project\artifacts\research_pipeline\evaluation\window_level_predictions.csv


## Step 1. Load Anomalous Windows From Evaluation Results

The adjudication stage starts from saved evaluation results. Only threshold-crossing windows are passed forward, which keeps the autoencoder as the sole anomaly detector.


In [2]:
eval_results = pd.read_csv(predictions_path)
for column in ["feature_error_vector", "top_feature_rank", "top_k_features", "top_k_feature_errors"]:
    if column in eval_results.columns:
        eval_results[column] = eval_results[column].apply(lambda value: literal_eval(value) if isinstance(value, str) else value)

ae_only_results = eval_results.copy()
ae_only_results["ae_only_label"] = ae_only_results["anomaly_score"].ge(threshold).map({True: "anomaly", False: "normal"})
anomalous_windows = ae_only_results[ae_only_results["ae_only_label"] == "anomaly"].copy().reset_index(drop=True)

print("Total evaluated windows:", len(ae_only_results))
print("Threshold-crossing windows:", len(anomalous_windows))
display(anomalous_windows[["window_id", "container_id", "machine_id", "anomaly_score"]].head(10))


Total evaluated windows: 24471
Threshold-crossing windows: 91


,window_id,container_id,machine_id,anomaly_score
0,70027,c_13275,m_2523,9.896001
1,153237,c_13590,m_811,11.704093
2,41984,c_26546,m_1903,16.877968
3,42663,c_26776,m_1924,19.119314
4,81839,c_29214,m_2808,15.489549
5,54191,c_10714,m_2208,8.515226
6,79829,c_16470,m_2742,6.135216
7,79989,c_10123,m_2754,5.534552
8,77045,c_11474,m_2672,7.473462
9,120697,c_21255,m_3708,8.371488


## Step 2. Compute Feature-Wise Reconstruction Errors

The evaluation output stores one reconstruction-error vector per window. Here the vector is expanded into explicit feature-wise columns so the notebook can inspect the contribution of each metric directly.


In [3]:
feature_error_columns = [f"{feature}_recon_error" for feature in feature_columns]
feature_error_df = pd.DataFrame(anomalous_windows["feature_error_vector"].tolist(), columns=feature_error_columns)
anomalous_windows = pd.concat([anomalous_windows.reset_index(drop=True), feature_error_df], axis=1)
display(anomalous_windows[["window_id", *feature_error_columns[: min(5, len(feature_error_columns))]]].head())


,window_id,cpu_util_recon_error,mem_util_recon_error,cpi_recon_error,mem_gps_recon_error,mpki_recon_error
0,70027,0.120228,0.058180,0.182428,78.189995,0.310343
1,153237,0.114857,0.028790,0.261172,92.661156,0.168912
2,41984,0.039647,0.034464,0.140370,134.351913,0.067187
3,42663,0.059830,0.039620,0.045759,152.293411,0.017377
4,81839,0.033783,0.072064,0.214572,122.952217,0.057935


## Step 3. Select Top-K Anomalous Features

The top-k anomalous features are chosen by ranking the feature-wise reconstruction errors for each threshold-crossing window.


In [4]:
top_k = config.gpt.top_k_features

def select_top_k_features(row: pd.Series) -> dict:
    ranked = sorted(
        [(feature, float(row[f"{feature}_recon_error"])) for feature in feature_columns],
        key=lambda item: item[1],
        reverse=True,
    )[:top_k]
    return {
        "top_k_features": [name for name, _ in ranked],
        "top_k_feature_errors": [score for _, score in ranked],
    }

if anomalous_windows.empty:
    anomalous_windows["top_k_features"] = pd.Series(dtype=object)
    anomalous_windows["top_k_feature_errors"] = pd.Series(dtype=object)
else:
    top_k_df = pd.DataFrame([select_top_k_features(row) for _, row in anomalous_windows.iterrows()])
    anomalous_windows["top_k_features"] = top_k_df["top_k_features"]
    anomalous_windows["top_k_feature_errors"] = top_k_df["top_k_feature_errors"]

display(anomalous_windows[["window_id", "top_k_features", "top_k_feature_errors"]].head())


,window_id,top_k_features,top_k_feature_errors
0,70027,"[mem_gps, mpki, disk_io, cpi, cpu_util]","[78.18999481201172, 0.31034258008003235, 0.248..."
1,153237,"[mem_gps, disk_io, cpi, mpki, cpu_util]","[92.6611557006836, 0.3770531415939331, 0.26117..."
2,41984,"[mem_gps, disk_io, cpi, mpki, cpu_util]","[134.35191345214844, 0.3202355206012726, 0.140..."
3,42663,"[mem_gps, disk_io, cpu_util, net_in, cpi]","[152.2934112548828, 0.4101439416408539, 0.0598..."
4,81839,"[mem_gps, disk_io, cpi, mem_util, net_in]","[122.95221710205078, 0.4650405943393707, 0.214..."


## Step 4. Build A Compact Prompt

Each prompt carries four compact inputs into GPT: the anomaly score, the top-k anomalous features, a context summary derived from Alibaba metadata, and optional recent logs from the runtime log file.


In [5]:
def read_recent_logs(log_path: Path, limit: int = 20) -> list[str]:
    if not log_path.exists():
        return []
    lines = [line.strip() for line in log_path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
    return lines[-limit:]

recent_logs = read_recent_logs(config.realtime.log_file)

prompt_payloads = []
for row in anomalous_windows.head(min(10, len(anomalous_windows))).itertuples(index=False):
    context_summary = {
        "container_id": row.container_id,
        "machine_id": row.machine_id,
        "app_du": getattr(row, "app_du", ""),
        "container_status": getattr(row, "container_status", ""),
        "machine_status": getattr(row, "machine_status", ""),
        "start_time": getattr(row, "start_time", ""),
        "end_time": getattr(row, "end_time", ""),
    }
    payload = {
        "window_id": int(row.window_id),
        "anomaly_score": float(row.anomaly_score),
        "threshold": float(threshold),
        "normalized_score": float(row.anomaly_score / max(threshold, 1e-6)),
        "top_k_anomalous_features": list(row.top_k_features),
        "top_k_feature_errors": list(row.top_k_feature_errors),
        "context_metadata": context_summary,
        "recent_logs": recent_logs,
        "recent_events": [],
    }
    payload["prompt_text"] = adjudicator.build_prompt(payload)
    prompt_payloads.append(payload)

sample_prompt = prompt_payloads[0]["prompt_text"] if prompt_payloads else "No anomalous windows available."
print(sample_prompt)


You are adjudicating a post-threshold multivariate container anomaly detected by an autoencoder.

Use the evidence below to decide whether this is a true anomaly, a false positive, or a case that needs review.

Return a JSON object that matches the provided schema. Do not include markdown or any extra commentary.

Scoring context:
- anomaly_score: 9.896001
- threshold: 5.500068187713623
- normalized_score: 1.7992506024027612

Context metadata:
{
  "container_id": "c_13275",
  "machine_id": "m_2523",
  "app_du": 505.0,
  "container_status": 0.0,
  "machine_status": 0.0,
  "start_time": 664120,
  "end_time": 680220
}

Top anomalous features:
[
  "mem_gps",
  "mpki",
  "disk_io",
  "cpi",
  "cpu_util"
]

Top anomalous feature errors:
[
  78.18999481201172,
  0.31034258008003235,
  0.248138427734375,
  0.1824280470609665,
  0.12022849917411804
]

Recent logs:
[]

Recent events:
[]

Decision rules:
- Use `false_positive` when the anomaly appears weak, noisy, or not operationally meaningful.

## Step 5. Call GPT

The notebook sends each post-threshold payload to the GPT adjudicator. If no API key is present, the module falls back to a deterministic heuristic so the notebook still runs end to end.


In [6]:
gpt_rows = []
for payload in prompt_payloads:
    decision = adjudicator.adjudicate(payload)
    gpt_rows.append({
        "window_id": payload["window_id"],
        "anomaly_score": payload["anomaly_score"],
        "container_id": payload["context_metadata"]["container_id"],
        "machine_id": payload["context_metadata"]["machine_id"],
        "normalized_score": payload["normalized_score"],
        "gpt_label": decision.get("label"),
        "severity": decision.get("severity"),
        "explanation": decision.get("explanation"),
        "recommended_action": decision.get("recommended_action"),
    })

gpt_results = pd.DataFrame(
    gpt_rows,
    columns=[
        "window_id",
        "anomaly_score",
        "container_id",
        "machine_id",
        "normalized_score",
        "gpt_label",
        "severity",
        "explanation",
        "recommended_action",
    ],
)
display(gpt_results.head())


,window_id,anomaly_score,container_id,machine_id,normalized_score,gpt_label,severity,explanation,recommended_action
0,70027,9.896001,c_13275,m_2523,1.799251,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
1,153237,11.704093,c_13590,m_811,2.127991,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
2,41984,16.877968,c_26546,m_1903,3.068683,anomaly,high,The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
3,42663,19.119314,c_26776,m_1924,3.476196,anomaly,high,The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
4,81839,15.489549,c_29214,m_2808,2.816247,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...


## Step 6. Parse Structured JSON Output

The GPT module returns a structured JSON object with the fields `label`, `severity`, `explanation`, and `recommended_action`. The parsed dataframe below exposes those fields alongside the originating anomaly metadata.


In [7]:
expected_columns = [
    "window_id",
    "container_id",
    "machine_id",
    "anomaly_score",
    "normalized_score",
    "gpt_label",
    "severity",
    "explanation",
    "recommended_action",
]
display(gpt_results[expected_columns].head() if not gpt_results.empty else gpt_results)


,window_id,container_id,machine_id,anomaly_score,normalized_score,gpt_label,severity,explanation,recommended_action
0,70027,c_13275,m_2523,9.896001,1.799251,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
1,153237,c_13590,m_811,11.704093,2.127991,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
2,41984,c_26546,m_1903,16.877968,3.068683,anomaly,high,The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
3,42663,c_26776,m_1924,19.119314,3.476196,anomaly,high,The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
4,81839,c_29214,m_2808,15.489549,2.816247,anomaly,medium,The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...


## Step 7. Compare AE-Only Labels Vs AE+GPT Labels

The final comparison keeps both decisions visible. The autoencoder determines whether a window crosses threshold, while GPT refines the post-threshold interpretation into anomaly, false positive, or review-needed outcomes.


In [8]:
comparison = anomalous_windows.merge(
    gpt_results,
    on=["window_id", "container_id", "machine_id"],
    how="left",
    suffixes=("", "_gpt"),
)
comparison = comparison[[
    "window_id",
    "container_id",
    "machine_id",
    "anomaly_score",
    "ae_only_label",
    "gpt_label",
    "severity",
    "top_k_features",
    "explanation",
    "recommended_action",
]]

display(comparison.head(10))

ae_gpt_paths = {
    "comparison_csv": str((config.paths.adjudication_dir / "ae_vs_gpt_comparison.csv").resolve()),
    "comparison_json": str((config.paths.adjudication_dir / "ae_vs_gpt_comparison.json").resolve()),
}
comparison.to_csv(config.paths.adjudication_dir / "ae_vs_gpt_comparison.csv", index=False)
save_json(config.paths.adjudication_dir / "ae_vs_gpt_comparison.json", {"records": comparison.to_dict(orient="records")})
display(pd.Series(ae_gpt_paths, name="saved_output"))


,window_id,container_id,machine_id,anomaly_score,ae_only_label,gpt_label,severity,top_k_features,explanation,recommended_action
0,70027,c_13275,m_2523,9.896001,anomaly,anomaly,medium,"[mem_gps, mpki, disk_io, cpi, cpu_util]",The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
1,153237,c_13590,m_811,11.704093,anomaly,anomaly,medium,"[mem_gps, disk_io, cpi, mpki, cpu_util]",The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
2,41984,c_26546,m_1903,16.877968,anomaly,anomaly,high,"[mem_gps, disk_io, cpi, mpki, cpu_util]",The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
3,42663,c_26776,m_1924,19.119314,anomaly,anomaly,high,"[mem_gps, disk_io, cpu_util, net_in, cpi]",The anomaly score is far above threshold and t...,"Treat this as an active incident, triage the a..."
4,81839,c_29214,m_2808,15.489549,anomaly,anomaly,medium,"[mem_gps, disk_io, cpi, mem_util, net_in]",The window shows a credible multivariate anoma...,Correlate the top metrics with service logs an...
5,54191,c_10714,m_2208,8.515226,anomaly,needs_review,low,"[mem_gps, disk_io, cpi, mem_util, cpu_util]",The anomaly is weak but persistent enough to r...,"Inspect the recent workload change, deployment..."
6,79829,c_16470,m_2742,6.135216,anomaly,needs_review,low,"[mem_gps, disk_io, mpki, cpu_util, cpi]",The anomaly is weak but persistent enough to r...,"Inspect the recent workload change, deployment..."
7,79989,c_10123,m_2754,5.534552,anomaly,false_positive,none,"[mem_gps, disk_io, cpi, mpki, net_in]",Reconstruction error only marginally exceeded ...,Keep monitoring and require repeated threshold...
8,77045,c_11474,m_2672,7.473462,anomaly,needs_review,low,"[mem_gps, disk_io, mpki, cpi, cpu_util]",The anomaly is weak but persistent enough to r...,"Inspect the recent workload change, deployment..."
9,120697,c_21255,m_3708,8.371488,anomaly,needs_review,low,"[mem_gps, disk_io, cpi, mpki, cpu_util]",The anomaly is weak but persistent enough to r...,"Inspect the recent workload change, deployment..."


comparison_csv     C:\Users\kaspe\Desktop\Project\project\artifac...
comparison_json    C:\Users\kaspe\Desktop\Project\project\artifac...
Name: saved_output, dtype: str